# 🧠 Customer Segmentation & Repayment Risk
**Author:** Gurupriya R | Mphasis Banking Analytics

K-Means clustering + Gradient Boosting to identify and flag high-risk banking customers.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')
df = pd.read_csv('data/customer_data.csv')
print(f'{len(df):,} customers | Risk dist:\n{df["repayment_risk"].value_counts()}')

## 1. Feature Engineering

In [ ]:
le = LabelEncoder()
df['employment_enc'] = le.fit_transform(df['employment_type'])
df['emi_to_income'] = df['monthly_emi'] / (df['annual_income']/12)
df['debt_to_income'] = df['outstanding_balance'] / df['annual_income']
df['payment_stress'] = df['num_missed_payments'] * df['emi_to_income']
FEATURES = ['credit_score','annual_income','outstanding_balance','monthly_emi',
            'num_missed_payments','total_loans','emi_to_income','debt_to_income','payment_stress']
print('Features:', FEATURES)

## 2. K-Means Clustering (k=4)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_scaled)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['pca_1'], df['pca_2'] = X_pca[:,0], X_pca[:,1]
fig, axes = plt.subplots(1,2,figsize=(14,5))
colors = ['#3a8c8c','#5ab0b0','#e8924a','#e07060']
for i in range(4):
    mask = df['segment']==i
    axes[0].scatter(df[mask]['pca_1'], df[mask]['pca_2'], c=colors[i], alpha=0.6, s=20, label=f'Seg {i}')
axes[0].set_title('Customer Segments (PCA)')
axes[0].legend()
df['segment'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Segment Sizes')
plt.tight_layout(); plt.show()

## 3. Repayment Risk Classification

In [ ]:
df['is_high_risk'] = (df['repayment_risk']=='High').astype(int)
X, y = df[FEATURES+['employment_enc']], df['is_high_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Standard','High-Risk']))

## 4. Flag High-Risk Accounts

In [ ]:
df['default_prob'] = gb.predict_proba(X)[:,1]
at_risk = df[df['default_prob']>0.6][['customer_id','credit_score','num_missed_payments','emi_to_income','default_prob']].sort_values('default_prob', ascending=False)
print(f'High-Risk Accounts: {len(at_risk)} ({len(at_risk)/len(df):.1%})')
at_risk.head(10)